# 01 — Data Ingestion
## Urban Mobility Intelligence Platform | Chicago TNC 2024

### Objective
Download all 12 months of 2024 Chicago Transportation Network Provider (TNC) trip data
from the City of Chicago Open Data Portal and load it into Google BigQuery.

### What this notebook does
1. Establishes connection to Google BigQuery
2. Creates the `chicago_tnc` dataset
3. Downloads each month as a CSV directly to disk (~1.1GB per month, ~14GB total)
4. Verifies all files are present and complete

### Output
- 12 CSV files saved to `data/raw_csv/`
- BigQuery dataset `chicago_tnc` created and ready for loading

### Dataset
- **Source:** City of Chicago Open Data Portal
- **Dataset ID:** aesv-xzh6
- **Period:** January 2024 – December 2024
- **Columns:** 14 (trip_id, timestamps, miles, seconds, zones, fares, shared trip flags)

In [ ]:

# IMPORTS & CONFIGURATION

import os
import time
import requests
import pandas as pd
from google.cloud import bigquery

# --- Credentials ---
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"E:\Projects\ML\Transport-Taxi-Chicago\gcp_credentials.json"

# --- BigQuery config ---
PROJECT_ID   = "urban-mobility-intel"
DATASET_ID   = "chicago_tnc"
TABLE_RAW    = f"{PROJECT_ID}.{DATASET_ID}.trips_2024_raw"
TABLE_FINAL  = f"{PROJECT_ID}.{DATASET_ID}.trips_final"

# --- Chicago Data Portal config ---
APP_TOKEN    = "OS7dIqMbjybMELmlM7jR00wDd"
SOCRATA_ID   = "aesv-xzh6"
BASE_URL     = f"https://data.cityofchicago.org/resource/{SOCRATA_ID}.csv"

# --- Local paths ---
RAW_CSV_DIR  = r"E:\Projects\ML\Transport-Taxi-Chicago\data\raw_csv"

# --- Columns we need (dropping centroid and census tract columns) ---
SELECTED_COLS = [
    "trip_id", "trip_start_timestamp", "trip_end_timestamp",
    "trip_seconds", "trip_miles", "pickup_community_area",
    "dropoff_community_area", "fare", "tip", "additional_charges",
    "trip_total", "shared_trip_authorized", "shared_trip_match",
    "trips_pooled"
]

print("Configuration loaded successfully.")
print(f"Project : {PROJECT_ID}")
print(f"Dataset : {DATASET_ID}")
print(f"CSV dir : {RAW_CSV_DIR}")

## 1. BigQuery Connection & Dataset Setup
Establish connection to BigQuery and create the `chicago_tnc` dataset if it doesn't exist.

In [ ]:
# BIGQUERY CONNECTION & DATASET SETUP

# Connect to BigQuery
client = bigquery.Client(project=PROJECT_ID)
print(f"Connected to BigQuery: {client.project}")

# Create dataset if it doesn't exist
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
dataset     = bigquery.Dataset(dataset_ref)
dataset.location = "US"

dataset = client.create_dataset(dataset, exists_ok=True)
print(f"Dataset ready: {dataset.dataset_id}")

## 2. Download Functions
Functions to download monthly CSVs directly to disk.
- Streams data in 1MB chunks — no RAM issues
- Downloads to `.tmp` file first — only renames to `.csv` when fully complete
- Retries up to 5 times with increasing wait time on failure
- Skips already completed months automatically

In [ ]:
# DOWNLOAD FUNCTIONS

def download_month_to_disk(year_month, start_date, end_date, output_path, max_retries=5):
    """
    Download a single month of TNC trip data directly to disk.
    
    Args:
        year_month  : Label e.g. '2024-01'
        start_date  : ISO format start e.g. '2024-01-01T00:00:00'
        end_date    : ISO format end   e.g. '2024-02-01T00:00:00'
        output_path : Full path to save CSV file
        max_retries : Number of retry attempts on failure
    
    Returns:
        True if successful, False if all retries failed
    """
    
    # Skip if already downloaded
    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"  {year_month} already exists ({size_mb:.1f} MB) — skipping.")
        return True
    
    # API parameters — select only needed columns, filter by date
    params = {
        "$select"  : ",".join(SELECTED_COLS),
        "$where"   : f"trip_start_timestamp >= '{start_date}' AND trip_start_timestamp < '{end_date}'",
        "$limit"   : 9999999,
        "$order"   : "trip_start_timestamp ASC",
        "$$app_token": APP_TOKEN
    }
    
    temp_path = output_path + ".tmp"
    
    for attempt in range(max_retries):
        try:
            print(f"  Downloading {year_month} (attempt {attempt+1}/{max_retries})...")
            
            # Stream directly to disk — no RAM loading
            with requests.get(BASE_URL, params=params, stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(temp_path, 'wb') as f:
                    downloaded = 0
                    for chunk in r.iter_content(chunk_size=1024*1024):  # 1MB chunks
                        f.write(chunk)
                        downloaded += len(chunk)
                        print(f"\r  {year_month}: {downloaded/1024/1024:.1f} MB downloaded", end="")
            
            # Only rename to .csv when fully complete — no partial files
            os.rename(temp_path, output_path)
            size_mb = os.path.getsize(output_path) / (1024 * 1024)
            print(f"\n  ✓ {year_month} saved ({size_mb:.1f} MB)")
            return True
            
        except Exception as e:
            print(f"\n  Error: {str(e)[:80]}")
            # Clean up partial temp file
            if os.path.exists(temp_path):
                os.remove(temp_path)
            if attempt < max_retries - 1:
                wait = 30 * (attempt + 1)
                print(f"  Waiting {wait}s before retry...")
                time.sleep(wait)
    
    print(f"  ✗ Failed after {max_retries} attempts.")
    return False

## 3. Download All 12 Months
Downloads each month sequentially to `data/raw_csv/`.
Already completed months are automatically skipped.
Expected total size: ~14GB across 12 files.

In [ ]:
# DOWNLOAD ALL 12 MONTHS

MONTHS = [
    ("2024-01", "2024-01-01T00:00:00", "2024-02-01T00:00:00"),
    ("2024-02", "2024-02-01T00:00:00", "2024-03-01T00:00:00"),
    ("2024-03", "2024-03-01T00:00:00", "2024-04-01T00:00:00"),
    ("2024-04", "2024-04-01T00:00:00", "2024-05-01T00:00:00"),
    ("2024-05", "2024-05-01T00:00:00", "2024-06-01T00:00:00"),
    ("2024-06", "2024-06-01T00:00:00", "2024-07-01T00:00:00"),
    ("2024-07", "2024-07-01T00:00:00", "2024-08-01T00:00:00"),
    ("2024-08", "2024-08-01T00:00:00", "2024-09-01T00:00:00"),
    ("2024-09", "2024-09-01T00:00:00", "2024-10-01T00:00:00"),
    ("2024-10", "2024-10-01T00:00:00", "2024-11-01T00:00:00"),
    ("2024-11", "2024-11-01T00:00:00", "2024-12-01T00:00:00"),
    ("2024-12", "2024-12-01T00:00:00", "2025-01-01T00:00:00"),
]

results = {}

for year_month, start_date, end_date in MONTHS:
    output_path = os.path.join(RAW_CSV_DIR, f"tnc_{year_month.replace('-', '_')}.csv")
    success = download_month_to_disk(year_month, start_date, end_date, output_path)
    results[year_month] = "✓" if success else "✗"

# Summary
print("\n--- Download Summary ---")
for month, status in results.items():
    print(f"  {status} {month}")

## 4. Verification
Verify all 12 files are present and check their sizes.
Expected: ~1.1GB per file, ~14GB total.

In [ ]:
# VERIFY DOWNLOADED FILES

print("--- File Verification ---\n")

total_size = 0
all_present = True

for year_month, _, _ in MONTHS:
    file_path = os.path.join(RAW_CSV_DIR, f"tnc_{year_month.replace('-', '_')}.csv")
    
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        total_size += size_mb
        print(f"  ✓ {year_month} — {size_mb:.1f} MB")
    else:
        print(f"  ✗ {year_month} — MISSING")
        all_present = False

print(f"\nTotal size : {total_size/1024:.2f} GB")
print(f"All present: {all_present}")